# Experiment Controller: Patient 020 Sanity Check

This notebook serves as a **thin experiment controller** for validating the Prostate MRI 2D segmentation pipeline on Patient 020.

> **Architecture Rule:** Core preprocessing, data handling, modeling, and evaluation logic reside in `code/` to maintain clean separation of concerns and reproducibility. This notebook coordinates execution and visual inspections.

## 1. Environment Setup
Add project root to Python path and verify essential libraries.

In [ ]:
import sys
import os

# Ensure project root is accessible for code imports
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import matplotlib.pyplot as plt
import nibabel as nib

print(f"Project root: {project_root}")
print(f"Python interpreter: {sys.executable}")

## 2. Configuration
Load configuration from `configs/config_020.yaml`.

In [ ]:
config_path = os.path.join(project_root, "configs", "config_020.yaml")

try:
    from code.utils import load_config
    config = load_config(config_path)
    print("Configuration successfully loaded:")
    for section, values in config.items():
        print(f"  [{section}]: {values}")
except ImportError:
    print("PyYAML not yet installed; install via requirements.txt to parse configs.")

## 3. Dataset Loading
Initialize `Prostate2DDataset` for Patient 020.

In [ ]:
from code.dataset import Prostate2DDataset

dataset_root = os.path.join(project_root, "dataset", "prostate158_train", "train")
dataset = Prostate2DDataset(
    dataset_root=dataset_root,
    patient_ids=["020"],
    modalities=["t2", "adc", "dwi"],
    mask_name="t2_tumor_reader1.nii.gz",
    slice_sampling="all"
)

print(f"Dataset initialized with {len(dataset)} 2D slices for Patient 020.")
sample = dataset[7]
print(f"Sample slice 7 image shape: {sample['image'].shape}, mask shape: {sample['mask'].shape}")

## 4. Visualization
Inspect axial slice 7 with multi-modal channels and tumor mask overlay.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
channel_names = ["T2-weighted", "ADC Map", "DWI", "Tumor Mask"]

for c in range(3):
    axes[c].imshow(sample['image'][c], cmap="gray")
    axes[c].set_title(channel_names[c])
    axes[c].axis("off")

axes[3].imshow(sample['image'][0], cmap="gray")
mask_overlay = np.ma.masked_where(sample['mask'][0] == 0, sample['mask'][0])
axes[3].imshow(mask_overlay, cmap="Reds", alpha=0.7)
axes[3].set_title("T2 + Tumor Mask Overlay")
axes[3].axis("off")

plt.tight_layout()
plt.show()

## 5. Model Initialization
Initialize 2D U-Net architecture placeholder.

In [ ]:
from code.model import ProstateUNet2D

model = ProstateUNet2D(
    in_channels=3,
    out_channels=1,
    init_features=32
)
print(f"Initialized model: {model.__class__.__name__}")
print("Note: Model forward pass is a placeholder awaiting PyTorch training implementation.")

## 6. Training Pipeline (Placeholder)
Controller hook to trigger training once fully implemented.

In [ ]:
# TODO: Wire train controller once PyTorch and training loop are implemented
print("Training pipeline controller ready for future execution.")
print("Command: python code/train.py configs/config_020.yaml")

## 7. Evaluation (Metrics Scaffolding)
Test metric calculation functions on ground truth.

In [ ]:
from code.evaluate import compute_dice, compute_iou

test_mask = sample['mask'][0]
# Perfect agreement self-test
dice_self = compute_dice(test_mask, test_mask)
iou_self = compute_iou(test_mask, test_mask)

print(f"Self-Dice Score: {dice_self:.4f}")
print(f"Self-IoU Score:  {iou_self:.4f}")

## 8. Results & Artifacts
Summary of existing verified outputs.

In [ ]:
inspection_report = os.path.join(project_root, "results", "020_dataset_inspection.md")
overview_png = os.path.join(project_root, "results", "020_patient_overview.png")

print(f"Dataset inspection report: {os.path.exists(inspection_report)} -> {inspection_report}")
print(f"Overview figure:           {os.path.exists(overview_png)} -> {overview_png}")